# Task 5 — Initial Descriptive Analysis

**Research question:** How accurately can classical regression models estimate engine Remaining Useful Life (RUL) from sensor data compared with a simple baseline?

Purpose: descriptive statistics, trends, hypotheses, grouping, and business implications. This is NOT a regression-model evaluation. Form hypotheses from training histories only; validation is reserved for descriptive replication, and test/quarantine data must not drive decisions. The retrospective RUL target is for analysis only, never a model input.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / "reports/results/task4/manifest.json").exists()), None)
if root is None:
    raise FileNotFoundError("Open this notebook from within the repository")
manifest = json.loads((root/"reports/results/task4/manifest.json").read_text())
file = root/"data/processed"/manifest["version_id"]/"flights_features.csv"
if not file.exists():
    raise FileNotFoundError(f"Run python scripts/prepare_data.py locally first: {file}")
all_flights = pd.read_csv(file)
assert len(all_flights) == manifest["processed_flights"]
train = all_flights.loc[all_flights.split == "train"].copy()
validation = all_flights.loc[all_flights.split == "validation"].copy()
assert set(train.conservative_group).isdisjoint(validation.conservative_group)
figdir = root/"reports/figures/task5"
outdir = root/"reports/results/task5"
figdir.mkdir(parents=True,exist_ok=True)
outdir.mkdir(parents=True,exist_ok=True)
sensors = ["T48_K_mean","Nc_rpm_mean","Wf_kg_s_mean"]
conditions = ["alt_m_mean","Mach_mean","TRA_pct_mean"]
print("Prepared version",manifest["version_id"],"training flights",len(train),"engines",train.engine_id.nunique())

## 1. Descriptive statistics and RUL distributions
Report engine-level and flight-level values separately: flights within one engine are dependent, not independent experimental subjects.

In [ ]:
summary=train[["RUL",*sensors,*conditions]].describe(percentiles=[.1,.25,.5,.75,.9]).T
summary.to_csv(outdir/"train_descriptive_statistics.csv")
display(summary)
per_engine=train.groupby(["dataset","engine_id"],observed=True).agg(flights=("RUL","size"),max_RUL=("RUL","max"),min_RUL=("RUL","min")).reset_index()
per_engine.to_csv(outdir/"train_engine_counts.csv",index=False)
display(per_engine)
fig,ax=plt.subplots(figsize=(8,4))
ax.hist(train.RUL,bins=24,color="#287b75",edgecolor="white")
ax.set(xlabel="RUL (flight cycles)",ylabel="Training flights",title="Training RUL distribution")
fig.tight_layout()
fig.savefig(figdir/"01_train_rul_histogram.svg")
plt.show()

## 2. Sensor trajectories by engine
Evaluate T48 temperature, core shaft speed Nc, and fuel flow Wf through the observed engine cycles. These signals are also affected by operating conditions; avoid causal claims.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,feature in zip(axes,sensors):
    for eid,g in train.groupby("engine_id"):
        g=g.sort_values("cycle")
        ax.plot(g.cycle,g[feature],alpha=.75,lw=1,label=eid)
    ax.set(xlabel="Engine flight cycle",ylabel=feature,title=feature)
axes[-1].legend(fontsize=6,ncol=2)
fig.suptitle("Training engine sensor trajectories")
fig.tight_layout()
fig.savefig(figdir/"02_sensor_trajectories.svg",bbox_inches="tight")
plt.show()

## 3. Sensor–RUL correlation and heterogeneity
H1: sensors vary with RUL. H4: relationships differ across engines/datasets. Pearson correlations summarize associations only; treat each engine history as a unit for comparison.

In [ ]:
features=["RUL",*sensors,*conditions,"T48_K_delta_prev","Nc_rpm_delta_prev","Wf_kg_s_delta_prev"]
corr=train[features].corr(numeric_only=True)
corr.to_csv(outdir/"train_correlation_matrix.csv")
fig,ax=plt.subplots(figsize=(9,7))
im=ax.imshow(corr,vmin=-1,vmax=1,cmap="coolwarm")
ax.set_xticks(range(len(features)),features,rotation=75,ha="right")
ax.set_yticks(range(len(features)),features)
fig.colorbar(im,ax=ax,label="Pearson r")
fig.tight_layout()
fig.savefig(figdir/"03_train_correlation_heatmap.svg",bbox_inches="tight")
plt.show()
rows=[]
for eid,g in train.groupby("engine_id"):
    for sensor in sensors:
        rows.append({"engine_id":eid,"sensor":sensor,"r_with_RUL":g[["RUL",sensor]].corr().iloc[0,1]})
engine_corr=pd.DataFrame(rows)
engine_corr.to_csv(outdir/"train_per_engine_correlations.csv",index=False)
display(engine_corr.pivot(index="engine_id",columns="sensor",values="r_with_RUL"))

## 4. Life-stage grouping (retrospective only)
Segment historical flights into Late RUL 0–20, Middle 21–50, Early >50. These fixed illustrative boundaries use the target label and must never be used as predictive model inputs or deployment logic. Aggregate first within each engine, then compare engines rather than treating flights as independent.

In [ ]:
train["life_stage"]=pd.cut(train.RUL,[-1,20,50,np.inf],labels=["Late","Middle","Early"])
medians=train.groupby(["dataset","engine_id","life_stage"],observed=True)[sensors].median().reset_index()
medians.to_csv(outdir/"train_life_stage_engine_medians.csv",index=False)
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,sensor in zip(axes,sensors):
    vals=[medians.loc[medians.life_stage==s,sensor].dropna() for s in ["Late","Middle","Early"]]
    ax.boxplot(vals,tick_labels=["Late","Middle","Early"])
    ax.set(xlabel="Historical RUL stage",ylabel=sensor,title=sensor)
fig.suptitle("Engine-level median sensor values by retrospective life stage")
fig.tight_layout()
fig.savefig(figdir/"04_life_stage_sensor_boxplots.svg",bbox_inches="tight")
plt.show()

## 5. Operating conditions and possible confounding
H3: altitude, Mach and throttle differ across life stages. Describe whether apparent sensor trends could reflect flight conditions, rather than degradation. These plots do not prove adjusted or causal effects.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,feature in zip(axes,conditions):
    vals=[train.loc[train.life_stage==s,feature].dropna() for s in ["Late","Middle","Early"]]
    ax.boxplot(vals,tick_labels=["Late","Middle","Early"],showfliers=False)
    ax.set(xlabel="Historical RUL stage",ylabel=feature,title=feature)
fig.suptitle("Operating conditions across historical RUL stages")
fig.tight_layout()
fig.savefig(figdir/"05_operating_conditions.svg",bbox_inches="tight")
plt.show()

## 6. H2: historical changes
Compare the descriptive behavior of current-flight means and previous-flight differences (T48, Nc and Wf). Previous-flight differences are absent for a history's first flight; this is expected. No feature-importance or incremental prediction claim is possible without training/evaluation.

In [ ]:
deltas=["T48_K_delta_prev","Nc_rpm_delta_prev","Wf_kg_s_delta_prev"]
delta_summary=train[deltas].describe().T
delta_summary.to_csv(outdir/"train_previous_flight_change_summary.csv")
display(delta_summary)
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,delta in zip(axes,deltas):
    v=train[delta].dropna()
    ax.hist(v,bins=30,color="#287b75",edgecolor="white")
    ax.set(xlabel=delta,ylabel="Training flights",title="Previous-flight change")
fig.tight_layout()
fig.savefig(figdir/"06_previous_flight_changes.svg",bbox_inches="tight")
plt.show()

## 7. Validation replication (after recording initial conclusions)
Review a small fixed set of per-engine sensor–RUL correlations on validation engines as a one-time consistency check. Do not inspect the test/quarantine split or adapt hypotheses repeatedly to validation.

In [ ]:
rows=[]
for eid,g in validation.groupby("engine_id"):
    for sensor in sensors:
        rows.append({"engine_id":eid,"sensor":sensor,"r_with_RUL":g[["RUL",sensor]].corr().iloc[0,1]})
validation_corr=pd.DataFrame(rows)
validation_corr.to_csv(outdir/"validation_per_engine_correlations.csv",index=False)
display(validation_corr.pivot(index="engine_id",columns="sensor",values="r_with_RUL"))

## 8. Key insights and business story — complete after execution

Record evidence for H1 sensor–RUL associations, H2 historical signal changes, H3 operating-condition confounding, and H4 engine-to-engine heterogeneity. For each include numerical result, exceptions, small number of independent engine histories, and implications for maintenance planning. Sensor measurements and degradation are simulated under real flight profiles. Do **not** claim measured maintenance savings, aviation suitability or baseline/regression accuracy.

**Next modeling stage:** compare a constant mean-training-RUL baseline with classical regressors using grouped validation; measure MAE/RMSE in cycles, per-engine errors, and optimistic RUL predictions. Keep test engines untouched until final evaluation.